In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("train.csv")

In [3]:
#جمع مقادیر خالی هر ستون
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [4]:
#درصد جمع مقادیر خالی هر ستون
(df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

Cabin          77.104377
Age            19.865320
Embarked        0.224467
PassengerId     0.000000
Name            0.000000
Pclass          0.000000
Survived        0.000000
Sex             0.000000
Parch           0.000000
SibSp           0.000000
Fare            0.000000
Ticket          0.000000
dtype: float64

In [5]:
#برای پرکردن مقادیر خالی سن از median استفاده میکنیم 
#چون سن ممکن است توزیع کاملاً نرمال نداشته باشد و median کمتر تحت تأثیر مقادیر پرت (مثل سن 80 سال) قرار می‌گیرد.
df["Age"] = df["Age"].fillna(df["Age"].median())

In [6]:
#برای تصمیم گیری برای ستون کابین نیازه ببینم این ستون چه مقادیری میگیره
df["Cabin"].value_counts()

Cabin
G6             4
C23 C25 C27    4
B96 B98        4
F33            3
E101           3
              ..
E17            1
A24            1
C50            1
B42            1
C148           1
Name: count, Length: 147, dtype: int64

In [7]:
#کابین بیشتر داده هاشو نداشتیم از 891 داده 687 تاش خالی بود
#چندتا کار میشه کرد اول اینکه یه فیچرجدید به اسم HasCabin  از مهندسی فیچر استفاده میکنم: ساخت
#بعدش میشه ستون Cabin رو حذف کرد
#یه کار دیگه قبل حذف ستون کابین میشه کرد اینه که 
#یه ستون جدید به اسم Deck ساخت و در اون دسته بندی کابین رو از ستون کابین دراورد گذاشت تو ستون جدید 
df["Deck"] = df["Cabin"].str[0] #از ستون Cabin، اولین کاراکتر (حرف اول) را بردار و داخل یک ستون جدید به نام Deck ذخیره کن.
df["Deck"] = df["Deck"].fillna("U") #مقادیر خالی ستون جدید رو پر کنیم 
#هر مقدار خالی (NaN) در ستون Deck را با حرف "U" جایگزین کن.
df.drop("Cabin", axis=1, inplace=True)#حالا میتونیم ستون کابین رو کلا حذف کنیم

In [8]:
#Embarked فقط دو مقدار خالی است.خیلی کم است، پس می‌توانیم با پرتکرارترین مقدار پر کنیم:
#اول ببینیم پرتکرارترین مقدار کدوم
df["Embarked"].value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [9]:
#پرتکرارترین مقدار S بود 
#حالا مقادیرخالی رو با S پر میکنیم
df["Embarked"] = df["Embarked"].fillna("S")

In [10]:
#دوباره چک میکنم ببین در کل ستون خالی ای باقی مونده که من فکری براش نکرده باشم
df.isnull().sum()

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
Deck           0
dtype: int64

In [11]:
#Name یکی از فیچرهای قوی تایتانیک است چون داخل اسم یک عنوان اجتماعی وجود داره که نشون دهنده جنسیت وضعیت تاهل و سن تقریبی است
#با استخراج عنوان اجتماعی (title)  میتونیم اطلاعات خوبی بگیریم 
# Extract title from Name
df["Title"] = df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)

# Remove original Name column
df.drop("Name", axis=1, inplace=True)

# Check titles
df["Title"].value_counts()

Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64

In [12]:
# بهتر است عنوان‌های کم‌تعداد را یکی کنیم
title_counts = df["Title"].value_counts()
rare_titles = title_counts[title_counts < 10].index #هر عنوان کمتر از ۱۰ بار را پیدا می‌کند و تبدیل به Rare می‌کند.
df["Title"] = df["Title"].replace(rare_titles, "Rare")

In [13]:
#چون مدل‌های Machine Learning با متن کار نمی‌کنند
###################################################################مرحله بعدی Encoding است؛
#ینی ستون های متن رو تبدیل به عدد میکنیم

In [14]:
#اول ببینیم ستون های معنی باقی مونده چیه 
df.select_dtypes(include=["object", "string"]).columns

Index(['Sex', 'Ticket', 'Embarked', 'Deck', 'Title'], dtype='str')

In [15]:

#اول از Sex شروع میکنیم چون فقط دومقدار میگیره
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

#بعدش میریم سراغ ستونهایی که مقادیر چند دسته ای میگیرن مثه Title, Deck,Embarked
#از روش  One-Hot Encoding استفاده میکنیم
df = pd.get_dummies(df, columns=["Embarked", "Title", "Deck"], drop_first=True)#ستون‌های متنی چنددسته‌ای را به ستون‌های عددی ۰ و ۱ تبدیل کن
#و برای جلوگیری از ستون‌های اضافی، یکی از دسته‌ها را حذف کن.
#drop_first=True برای جلوگیری از ایجاد ستون‌های اضافی و مشکل multicollinearity است.

#فعلا ستون Ticket رو حذف میکنیم
#چون تیکت خام مثل: 347082 PC 17599 STON/O2. 3101282 اطلاعات مستقیم قابل استفاده‌ای برای مدل ندارد.
df.drop("Ticket", axis=1, inplace=True)

In [16]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch',
       'Fare', 'Embarked_Q', 'Embarked_S', 'Title_Miss', 'Title_Mr',
       'Title_Mrs', 'Title_Rare', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E',
       'Deck_F', 'Deck_G', 'Deck_T', 'Deck_U'],
      dtype='str')

In [17]:
#PassengerId معمولاً Feature خوبی نیست، چون فقط شماره ردیف مسافر است و اطلاعات واقعی درباره بقا نمی‌دهد. حذفش می‌کنیم:
df.drop("PassengerId", axis=1, inplace=True)

In [18]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1 #عدد یک واسه اینه که خود شخص رو هم بشماریم

In [19]:
#تو نوت بوک بعد میریم سراغ انتخاب مدل و تنظیم پارامترا و بهبود مدل